# Organização
algorithms.py
    → genetic_algorithm()
    → pso()

parkinsons_problem.py
    → generate_solution()
    → fitness_function()

utils_nn.py
    → vector_to_weights()
    → set_mlp_weights()
    → count_parameters()

main.py
    → run everything

### Teorical  Why we turn everything into a vector

Instead of storing:

W1 = matrix (22 x 10)
b1 = vector (10)
W2 = matrix (10 x 1)
b2 = scalar

We flatten everything into:

[0.2, -0.5, 1.3, ..., 0.7]

Why?
Because optimization algorithms (GA, PSO) work on vectors.

MLPClassifier expects weights in a VERY specific format

It does NOT accept:

[0.2, -0.5, 1.3, ..., 0.7]   # ❌ flat vector

Instead, it expects:

coefs = [W1, W2]
intercepts = [b1, b2]

vector → split → reshape → give to MLP

W1[i][j] = how much input i influences neuron j


1. GA gives you a vector
2. You convert it into (W1, b1, W2, b2)
3. You plug it into MLPClassifier
4. You run predictions
5. You measure performance

# DATA
This dataset is composed of a range of biomedical voice measurements from **31 people, 23 with Parkinson's disease (PD)**. Each column in the table is a particular voice measure, and each row corresponds to one of 195 voice recordings from these individuals ("name" column). The main aim of the data is to **discriminate healthy people from those with PD**, according to the "status" column which is set to 0 for healthy and 1 for PD.

## Attribute Information:
### Matrix column entries (attributes):
- name - ASCII subject name and recording number
- MDVP:Fo(Hz) - Average vocal fundamental frequency
- MDVP:Fhi(Hz) - Maximum vocal fundamental frequency
- MDVP:Flo(Hz) - Minimum vocal fundamental frequency
- MDVP:Jitter(%), MDVP:Jitter(Abs), MDVP:RAP, MDVP:PPQ, Jitter:DDP - Several measures of variation in fundamental frequency
- MDVP:Shimmer,MDVP:Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,MDVP:APQ,Shimmer:DDA - Several measures of variation in amplitude
- NHR, HNR - Two measures of the ratio of noise to tonal components in the voice
- status - The health status of the subject (one) - Parkinson's, (zero) - healthy
- RPDE, D2 - Two nonlinear dynamical complexity measures
- DFA - Signal fractal scaling exponent
- spread1,spread2,PPE - Three nonlinear measures of fundamental frequency variation

# Attribute Information:
## Matrix column entries (attributes):
- name - ASCII subject name and recording number
- MDVP:Fo(Hz) - Average vocal fundamental frequency
- MDVP:Fhi(Hz) - Maximum vocal fundamental frequency
- MDVP:Flo(Hz) - Minimum vocal fundamental frequency
- MDVP:Jitter(%), MDVP:Jitter(Abs), MDVP:RAP, MDVP:PPQ, Jitter:DDP - Several measures of variation in fundamental frequency
- MDVP:Shimmer,MDVP:Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,MDVP:APQ,Shimmer:DDA - Several measures of variation in amplitude
- NHR, HNR - Two measures of the ratio of noise to tonal components in the voice
- status - The health status of the subject (one) - Parkinson's, (zero) - healthy
- RPDE, D2 - Two nonlinear dynamical complexity measures
- DFA - Signal fractal scaling exponent
- spread1,spread2,PPE - Three nonlinear measures of fundamental frequency variation

# Justificação das Escolhas do Projeto de Otimização

## 1️⃣ Inicialização (Initialization)

- **Escolhidas:** `random_initialisation` e `xavier_initialisation`.
- **Por que?**
  - `random_initialisation`: garante **diversidade inicial**, explorando amplamente o espaço de soluções.
  - `xavier_initialisation`: ajusta os pesos de acordo com a estrutura da rede, evitando saturação de neurônios. População inicial mais próxima de regiões de boas soluções → convergência mais rápida e eficiente:contentReference[oaicite:0]{index=0}.
- **Alternativas não usadas:**
  - `zeros` ou valores constantes → quebrariam a diversidade, levando a população a regiões ruins.
  - `He-initialization` → projetada para ReLU profundas; para nossa MLP pequena Xavier é suficiente e mais estável.

---

## 2️⃣ Seleção (Selection)

- **Escolhidas:** `tournament_selection` e `roulette_wheel_selection`.
- **Por que?**
  - `tournament_selection`: fácil de controlar a pressão seletiva pelo tamanho do torneio; mantém diversidade e evita convergência prematura.
  - `roulette_wheel_selection`: fitness-proportionate, todos têm chance de reprodução, favorece exploração de soluções intermediárias:contentReference[oaicite:1]{index=1}.
- **Alternativas não usadas:**
  - `rank-based selection` → complexo para interpretar resultados e ajustar para nosso problema clínico.
  - `stochastic universal sampling` → sofisticado, pouco benefício adicional para a nossa população de tamanho moderado.

---

## 3️⃣ Crossover

- **Escolhidos:** `arithmetic_crossover` e `blend_crossover (BLX-α)`.
- **Por que?**
  - `arithmetic_crossover`: gera filhos como **interpolação entre pais**; bom para pesos reais e landscapes contínuos. Filhos intermediários têm boa chance de manter performance dos pais.
  - `blend_crossover (BLX-α)`: permite **extrapolar além dos pais**, explorando regiões nunca visitadas e ajudando a escapar de ótimos locais:contentReference[oaicite:2]{index=2}.
- **Alternativas não usadas:**
  - Crossover binário (1-point, 2-point) → adequado para cromossomos binários, não para vetores de pesos contínuos.
  - Crossover uniforme → poderia ser usado, mas mistura genes arbitrariamente sem respeitar relação entre valores contínuos, o que pode gerar filhos com desempenho muito ruim.

---

## 4️⃣ Mutação

- **Escolhidas:** `gaussian_mutation` e `uniform_reset_mutation`.
- **Por que?**
  - `gaussian_mutation`: faz **pequenas perturbações**, ideal para **refinar soluções** já próximas do ótimo.
  - `uniform_reset_mutation`: reseta aleatoriamente alguns genes, permitindo **exploração de novas regiões** e evitando estagnação da população:contentReference[oaicite:3]{index=3}.
- **Alternativas não usadas:**
  - Mutação baseada em outros ruídos (e.g., Cauchy) → poderia gerar saltos muito grandes ou instabilidade.
  - Mutação adaptativa complexa → requer mais tuning e traz pouco benefício no nosso problema de dimensão moderada.

---

## 5️⃣ Fitness / Métrica

- **Escolhida:** **Recall da classe positiva (Parkinson’s)**.
- **Por que?**
  - Dataset desbalanceado: 147 doentes vs 48 saudáveis (≈75% positivos):contentReference[oaicite:4]{index=4}.
  - Objetivo clínico: **minimizar falsos negativos** (não deixar nenhum paciente doente sem detecção).
- **Complementarmente:** reportamos **Precision, Accuracy e F1** para análise completa e comparação.

---

## 6️⃣ Algoritmos usados

- **GA (Genetic Algorithm):** explora múltiplas regiões, combina e muta soluções, adequado para problemas de alta dimensão e landscape contínuo:contentReference[oaicite:5]{index=5}:contentReference[oaicite:6]{index=6}.
- **PSO (Particle Swarm Optimization):** inspirado em comportamento coletivo (pássaros/peixes), cada partícula segue experiência própria e do grupo → balanceia exploração e explotação:contentReference[oaicite:7]{index=7}.

---

## 7️⃣ Resumo de escolhas estratégicas

| Componente      | Escolha           | Por que? | Alternativas não usadas e por que não foram escolhidas |
|-----------------|-----------------|----------|------------------------------------------------------|
| Inicialização   | random / Xavier  | diversidade + proximidade de boas soluções | zeros/constantes (perda de diversidade), He-init (desnecessário para MLP pequena) |
| Seleção         | tournament / roulette | mantém diversidade e pressão seletiva controlada | rank-based (mais complexo), stochastic universal sampling (benefício marginal) |
| Crossover       | arithmetic / BLX-α | interpolação + extrapolação, adequado para pesos reais | binário (não contínuo), uniforme (pode gerar filhos ruins) |
| Mutação         | gaussian / uniform reset | refine + explore, evita estagnação | outros ruídos (Cauchy, adaptativos) → instáveis ou complexos |
| Fitness         | Recall           | minimiza FN, prioridade clínica | Accuracy (enganosa em imbalance), Precision (ignora FN), F1 pode ser secundário |

---

Se quiser, posso criar **uma versão visual em markdown** mostrando o **fluxo GA + PSO** (população → seleção → crossover → mutação → fitness Recall) que você pode colocar no notebook e fica ótimo para a apresentação.

Quer que eu faça isso agora?